In [8]:
from openai import OpenAI
import json, time
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
client = OpenAI()

In [11]:
import time

def call_llm(prompt, retries=3):

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                 model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
               
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            error = str(e)

            # Auth error — retrying won't help, fix key first
            if "401" in error or "authentication" in error.lower():
                print("❌ Auth error — check your API key in .env")
                return "AUTH_ERROR"

            # Rate limit or server error — wait and retry
            wait = 2 ** attempt   # 1s → 2s → 4s
            print(f"⚠️  Attempt {attempt+1}/{retries} failed."
                  f" Waiting {wait}s... ({error[:50]})")
            time.sleep(wait)


In [12]:
call_llm('hello')

'Hello! How can I assist you today?'

In [16]:
print('waiting for llm call')
time.sleep(10)
call_llm('hello again')

waiting for llm call


'Hello! Welcome back! How can I assist you today?'

In [ ]:
age = int(input('enter your age'))

try:
    age = age/0
except ZeroDivisionError as e:
    print("Error: Division by zero is not allowed.")

Error: Division by zero is not allowed.


In [3]:
age = age/0

ZeroDivisionError: division by zero

In [18]:
import hashlib

# In-memory cache — survives for the session
# For production: use Redis or a database
_cache       = {}
_cache_hits  = 0
_cache_misses = 0

def get_cache_key(email):
    """
    Create unique fingerprint for each email.
    Same subject + body (after normalising) = same key = cache hit.
    Why MD5? Fast, deterministic, not for security here.
    """
    content = (
        email['subject'].lower().strip() + "|" +
        email['body'].lower().strip()
    )
    return hashlib.md5(content.encode()).hexdigest()


def classify_with_cache(email, prompt_template):
    """
    Check cache first. Call API only on cache miss.
    Cache hit = instant response + zero cost.
    Cache miss = API call + store result for next time.
    """
    global _cache_hits, _cache_misses

    key = get_cache_key(email)

    # Cache HIT — return stored result instantly
    if key in _cache:
        _cache_hits += 1
        print(f"  [CACHE HIT]  No API call. Saved $0.000135")
        return {**_cache[key], "source": "CACHE"}

    # Cache MISS — call the API
    _cache_misses += 1
    subject = email['subject']
    body    = email['body']
    prompt  = prompt_template.format(subject=subject, body=body)
    result  = call_llm(prompt)

    # Store in cache for all future requests
    output = {"category": result.strip()}
    _cache[key] = output
    print(f"  [CACHE MISS] API called. Cached for future.")
    return {**output, "source": "API"}


# Test — 5 emails, 2 duplicates
test_emails = [
    {"id": 1, "subject": "Refund policy?",
     "body": "Please tell me the refund policy for annual plan."},
    {"id": 2, "subject": "App crash on settings",
     "body": "App crashes when I open Settings page. Chrome 120."},
    {"id": 3, "subject": "Refund policy?",  # exact duplicate!
     "body": "Please tell me the refund policy for annual plan."},
    {"id": 4, "subject": "SFDC not syncing",
     "body": "Salesforce integration stopped pulling leads. 3 days."},
    {"id": 5, "subject": "Refund policy?",  # another duplicate!
     "body": "Please tell me the refund policy for annual plan."},
]

PROMPT = """Return ONLY category: Billing | Technical | Feature Request | Spam | Other
Subject: {subject}
Body: {body}"""

print("=" * 55)
print("CACHING TEST")
print("=" * 55)

for email in test_emails:
    result = classify_with_cache(email, PROMPT)
    print(f"Email {email['id']}: {email['subject'][:35]:35} → {result['category']}")

print()
print(f"API calls made : {_cache_misses}  (you paid for these)")
print(f"Cache hits     : {_cache_hits}   (completely free)")
print(f"Cost saved     : ${_cache_hits * 0.000135:.6f}")

CACHING TEST
  [CACHE MISS] API called. Cached for future.
Email 1: Refund policy?                      → Billing
  [CACHE MISS] API called. Cached for future.
Email 2: App crash on settings               → Technical
  [CACHE HIT]  No API call. Saved $0.000135
Email 3: Refund policy?                      → Billing
  [CACHE MISS] API called. Cached for future.
Email 4: SFDC not syncing                    → Technical
  [CACHE HIT]  No API call. Saved $0.000135
Email 5: Refund policy?                      → Billing

API calls made : 3  (you paid for these)
Cache hits     : 2   (completely free)
Cost saved     : $0.000270


In [20]:
import logging

# Setup once at top of your pipeline file
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('pipeline.log'),  # writes to file
        logging.StreamHandler()                  # also prints to terminal
    ]
)
logger = logging.getLogger('email_pipeline')

# Use in your pipeline:
logger.info(f"Processing email 47: 'Can't login...'")
logger.info(f"Cache hit — no API call")
logger.info(f"Classified: Billing | High urgency")
logger.info(f"Routed to: billing@company.com")
logger.warning(f"Email rejected: Injection detected")
logger.error(f"API error on attempt 3: RateLimitError")

2026-04-19 12:28:27,589 | INFO | Processing email 47: 'Can't login...'
2026-04-19 12:28:27,592 | INFO | Cache hit — no API call
2026-04-19 12:28:27,593 | INFO | Classified: Billing | High urgency
2026-04-19 12:28:27,595 | INFO | Routed to: billing@company.com
2026-04-19 12:28:27,598 | WARNING | Email rejected: Injection detected
2026-04-19 12:28:27,605 | ERROR | API error on attempt 3: RateLimitError
